# Semana 9: Aprendizaje No Supervisado: Clustering y PCA
## Notebook Conceptual (NB1) – Visualización de Componentes y Clusters

**Propósito:** Introducir técnicas para datos no etiquetados: reducción de dimensionalidad (PCA) y clustering (K-Means, DBSCAN).

**Docente:** Carlos César Sánchez Coronel

**Objetivos de aprendizaje:**
- Calcular autovalores y autovectores de la matriz de covarianza e implementar PCA desde cero.
- Visualizar la varianza explicada y proyectar datos en 2D.
- Implementar K-Means paso a paso y visualizar la evolución de centroides.
- Aplicar el método del codo y el coeficiente de silueta para elegir k.
- Experimentar con DBSCAN y visualizar el efecto de eps y minPts.
- Comparar K-Means y DBSCAN en datos con formas no esféricas.

---

## 0. Configuración Inicial

Importamos las librerías necesarias y fijamos la semilla para reproducibilidad.

In [ ]:
# Importamos librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D
from sklearn.datasets import make_blobs, make_moons, make_circles
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, silhouette_samples

# Configuración de visualización
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# Semilla
np.random.seed(42)

print("Librerías importadas correctamente.")

---
## 1. Generación de Datos Sintéticos

Creamos diferentes tipos de datasets para explorar las técnicas:
1. **Blobs** (clusters esféricos) en 3D para PCA
2. **Moons** (formas no lineales) para comparar K-Means y DBSCAN
3. **Círculos concéntricos** para DBSCAN

In [ ]:
# 1. Blobs en 3D (para PCA)
X_blobs_3d, y_blobs_3d = make_blobs(n_samples=300, n_features=3, centers=4, 
                                     cluster_std=1.5, random_state=42)

# 2. Moons (para clustering)
X_moons, y_moons = make_moons(n_samples=300, noise=0.05, random_state=42)

# 3. Círculos concéntricos (para DBSCAN)
X_circles, y_circles = make_circles(n_samples=300, noise=0.05, factor=0.3, random_state=42)

# Visualizamos
fig = plt.figure(figsize=(15, 4))

ax1 = fig.add_subplot(131, projection='3d')
ax1.scatter(X_blobs_3d[:, 0], X_blobs_3d[:, 1], X_blobs_3d[:, 2], c=y_blobs_3d, cmap='viridis', alpha=0.6)
ax1.set_title('Blobs en 3D')

ax2 = fig.add_subplot(132)
ax2.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap='viridis', alpha=0.6)
ax2.set_title('Moons')

ax3 = fig.add_subplot(133)
ax3.scatter(X_circles[:, 0], X_circles[:, 1], c=y_circles, cmap='viridis', alpha=0.6)
ax3.set_title('Círculos concéntricos')

plt.tight_layout()
plt.show()

---
## 2. PCA: Implementación Manual

Implementamos PCA desde cero:
1. Centrar los datos
2. Calcular la matriz de covarianza
3. Calcular autovalores y autovectores
4. Proyectar los datos

In [ ]:
# Centramos los datos
X_centered = X_blobs_3d - np.mean(X_blobs_3d, axis=0)

# Matriz de covarianza
cov_matrix = np.cov(X_centered.T)
print("Matriz de covarianza:\n", cov_matrix)

# Autovalores y autovectores
eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)

# Ordenamos por autovalor descendente
idx = eigenvalues.argsort()[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

print("\nAutovalores:", eigenvalues)
print("Autovectores:\n", eigenvectors)

# Varianza explicada
explained_variance_ratio = eigenvalues / np.sum(eigenvalues)
print("\nProporción de varianza explicada:", explained_variance_ratio)

# Proyectamos a 2D
X_pca_manual = X_centered @ eigenvectors[:, :2]

# Visualizamos
plt.figure(figsize=(8, 6))
plt.scatter(X_pca_manual[:, 0], X_pca_manual[:, 1], c=y_blobs_3d, cmap='viridis', alpha=0.6)
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.title('PCA Manual: Proyección a 2D')
plt.colorbar(label='Cluster real')
plt.show()

### 2.1. Comparación con PCA de scikit-learn

In [ ]:
pca = PCA(n_components=2)
X_pca_sk = pca.fit_transform(X_blobs_3d)

print("Varianza explicada (sklearn):", pca.explained_variance_ratio_)

# Scree plot
plt.figure(figsize=(8, 4))
plt.bar(range(1, 4), pca.explained_variance_ratio_, alpha=0.7)
plt.step(range(1, 4), np.cumsum(pca.explained_variance_ratio_), where='mid', color='red')
plt.xlabel('Componente Principal')
plt.ylabel('Proporción de Varianza Explicada')
plt.title('Scree Plot')
plt.show()

---
## 3. K-Means: Implementación Paso a Paso

Implementamos K-Means manualmente y visualizamos la evolución de los centroides.

In [ ]:
class KMeansManual:
    def __init__(self, k=3, max_iters=10, random_state=42):
        self.k = k
        self.max_iters = max_iters
        self.random_state = random_state
        self.centroids = None
        self.labels = None
        self.history = []  # para guardar evolución
    
    def fit(self, X):
        np.random.seed(self.random_state)
        # Inicialización aleatoria de centroides (seleccionando puntos aleatorios)
        idx = np.random.choice(X.shape[0], self.k, replace=False)
        self.centroids = X[idx].copy()
        self.history.append(self.centroids.copy())
        
        for _ in range(self.max_iters):
            # Asignación: calcular distancia a cada centroide
            distances = np.linalg.norm(X[:, np.newaxis] - self.centroids, axis=2)
            self.labels = np.argmin(distances, axis=1)
            
            # Actualización: recalcular centroides
            new_centroids = np.array([X[self.labels == i].mean(axis=0) for i in range(self.k)])
            
            # Verificar convergencia
            if np.allclose(self.centroids, new_centroids):
                break
            
            self.centroids = new_centroids
            self.history.append(self.centroids.copy())
        
        return self
    
    def predict(self, X):
        distances = np.linalg.norm(X[:, np.newaxis] - self.centroids, axis=2)
        return np.argmin(distances, axis=1)

# Aplicamos a los blobs (en 2D para facilitar visualización)
X_blobs_2d, _ = make_blobs(n_samples=200, centers=3, cluster_std=1.0, random_state=42)

kmeans_manual = KMeansManual(k=3, max_iters=10)
kmeans_manual.fit(X_blobs_2d)

### 3.1. Visualización de la evolución de centroides

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.ravel()

for i, centroids in enumerate(kmeans_manual.history):
    axes[i].scatter(X_blobs_2d[:, 0], X_blobs_2d[:, 1], alpha=0.3)
    axes[i].scatter(centroids[:, 0], centroids[:, 1], c='red', s=100, marker='X', label='Centroides')
    axes[i].set_title(f'Iteración {i}')
    axes[i].legend()

# Ocultar subplots vacíos
for i in range(len(kmeans_manual.history), len(axes)):
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## 4. Método del Codo y Coeficiente de Silueta

Usamos los blobs 2D para encontrar el mejor k.

In [ ]:
inertias = []
silhouettes = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_blobs_2d)
    inertias.append(kmeans.inertia_)
    silhouettes.append(silhouette_score(X_blobs_2d, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(K_range, inertias, 'bo-')
axes[0].set_xlabel('k')
axes[0].set_ylabel('Inercia')
axes[0].set_title('Método del Codo')
axes[0].grid(True)

axes[1].plot(K_range, silhouettes, 'ro-')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Coeficiente de Silueta')
axes[1].set_title('Silueta vs k')
axes[1].grid(True)

plt.tight_layout()
plt.show()

best_k = K_range[np.argmax(silhouettes)]
print(f"Mejor k según silueta: {best_k}")

### 4.1. Visualización de la silueta para el mejor k

In [ ]:
kmeans_best = KMeans(n_clusters=best_k, random_state=42, n_init=10)
labels_best = kmeans_best.fit_predict(X_blobs_2d)

silhouette_vals = silhouette_samples(X_blobs_2d, labels_best)

fig, ax = plt.subplots(figsize=(10, 6))

y_lower = 10
for i in range(best_k):
    ith_cluster_silhouette_vals = silhouette_vals[labels_best == i]
    ith_cluster_silhouette_vals.sort()
    
    size_cluster_i = ith_cluster_silhouette_vals.shape[0]
    y_upper = y_lower + size_cluster_i
    
    color = plt.cm.nipy_spectral(float(i) / best_k)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, ith_cluster_silhouette_vals,
                     facecolor=color, edgecolor=color, alpha=0.7)
    
    ax.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))
    y_lower = y_upper + 10

ax.axvline(x=silhouette_vals.mean(), color='red', linestyle='--', label='Media de silueta')
ax.set_xlabel('Coeficiente de silueta')
ax.set_ylabel('Cluster')
ax.set_title('Gráfico de silueta para k = {}'.format(best_k))
ax.legend()
plt.show()

---
## 5. DBSCAN: Efecto de eps y minPts

Aplicamos DBSCAN a los datos de círculos y lunas para ver cómo afectan los parámetros.

In [ ]:
def plot_dbscan_parameters(X, eps_values, min_samples_values, title_prefix):
    fig, axes = plt.subplots(len(eps_values), len(min_samples_values), 
                              figsize=(15, 10))
    
    for i, eps in enumerate(eps_values):
        for j, min_samples in enumerate(min_samples_values):
            dbscan = DBSCAN(eps=eps, min_samples=min_samples)
            labels = dbscan.fit_predict(X)
            
            n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
            n_noise = list(labels).count(-1)
            
            axes[i, j].scatter(X[:, 0], X[:, 1], c=labels, cmap='viridis', 
                               alpha=0.6, edgecolors='k', s=30)
            axes[i, j].set_title(f'eps={eps}, min_samples={min_samples}\nClusters={n_clusters}, Ruido={n_noise}')
            axes[i, j].set_xticks([])
            axes[i, j].set_yticks([])
    
    plt.suptitle(title_prefix)
    plt.tight_layout()
    plt.show()

# Probamos diferentes combinaciones en círculos
eps_values = [0.1, 0.2, 0.3]
min_samples_values = [3, 5, 10]
plot_dbscan_parameters(X_circles, eps_values, min_samples_values, 'DBSCAN en círculos')

In [ ]:
# Ahora en lunas
plot_dbscan_parameters(X_moons, eps_values, min_samples_values, 'DBSCAN en lunas')

---
## 6. Comparación: K-Means vs DBSCAN en Datos No Esféricos

Comparamos el rendimiento de K-Means y DBSCAN en los datos de lunas.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Datos originales
axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c='gray', alpha=0.6)
axes[0].set_title('Datos originales (lunas)')

# K-Means (k=2)
kmeans_moons = KMeans(n_clusters=2, random_state=42)
labels_kmeans = kmeans_moons.fit_predict(X_moons)
axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=labels_kmeans, cmap='viridis', alpha=0.6)
axes[1].scatter(kmeans_moons.cluster_centers_[:, 0], kmeans_moons.cluster_centers_[:, 1], 
                c='red', s=200, marker='X', label='Centroides')
axes[1].set_title('K-Means (k=2)')
axes[1].legend()

# DBSCAN
dbscan_moons = DBSCAN(eps=0.2, min_samples=5)
labels_dbscan = dbscan_moons.fit_predict(X_moons)
axes[2].scatter(X_moons[:, 0], X_moons[:, 1], c=labels_dbscan, cmap='viridis', alpha=0.6)
axes[2].set_title('DBSCAN (eps=0.2, min_samples=5)')

plt.tight_layout()
plt.show()

print(f"K-Means: {len(set(labels_kmeans))} clusters")
print(f"DBSCAN: {len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)} clusters, {list(labels_dbscan).count(-1)} puntos de ruido")

---
## 7. Experimentación Adicional: K-Means en formas no esféricas

Mostramos las limitaciones de K-Means en datos con formas complejas.

In [ ]:
# Datos con formas no esféricas
X_shapes, y_shapes = make_circles(n_samples=300, noise=0.05, factor=0.3, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

kmeans_shapes = KMeans(n_clusters=2, random_state=42)
labels_shapes = kmeans_shapes.fit_predict(X_shapes)

axes[0].scatter(X_shapes[:, 0], X_shapes[:, 1], c=labels_shapes, cmap='viridis', alpha=0.6)
axes[0].scatter(kmeans_shapes.cluster_centers_[:, 0], kmeans_shapes.cluster_centers_[:, 1], 
                c='red', s=200, marker='X')
axes[0].set_title('K-Means en círculos concéntricos')

dbscan_shapes = DBSCAN(eps=0.15, min_samples=5)
labels_dbscan_shapes = dbscan_shapes.fit_predict(X_shapes)

axes[1].scatter(X_shapes[:, 0], X_shapes[:, 1], c=labels_dbscan_shapes, cmap='viridis', alpha=0.6)
axes[1].set_title('DBSCAN en círculos concéntricos')

plt.tight_layout()
plt.show()

---
## 8. Conclusiones

Hemos explorado las técnicas fundamentales de aprendizaje no supervisado:

✔️ **PCA**:
- Implementación manual desde la matriz de covarianza.
- Visualización de la varianza explicada (scree plot).
- Proyección de datos 3D a 2D preservando la máxima varianza.

✔️ **K-Means**:
- Implementación paso a paso con visualización de la evolución de centroides.
- Método del codo y coeficiente de silueta para elegir k.
- Gráfico de silueta para evaluar la calidad de los clusters.

✔️ **DBSCAN**:
- Efecto de los parámetros eps y minPts en la formación de clusters.
- Capacidad de detectar ruido y clusters de forma arbitraria.

✔️ **Comparación**:
- K-Means funciona bien en clusters esféricos, pero falla en formas no lineales.
- DBSCAN es más flexible pero requiere ajuste cuidadoso de parámetros.

**Lección clave**: La elección del algoritmo y sus parámetros depende de la estructura de los datos. El análisis visual y las métricas internas (silueta, inercia) guían la selección.

En el próximo notebook (NB2) aplicaremos estos conceptos a un problema real de segmentación de clientes.

---
**Fin del Notebook Conceptual - Semana 9**